# Wafer Overlay Modeling — Colab notebook

Self-contained version: no extra files needed.

**Run:** `Runtime → Run all` (or run each cell top to bottom with Shift+Enter).

Plots and the summary are saved in the `results/` folder (folder icon on the left).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

RESULTS = "results"
os.makedirs(RESULTS, exist_ok=True)
print('numpy', np.__version__)

## 1. Core functions
Layout (Lesson 1), simulation (Lesson 2), least-squares fit (Lesson 3), robust fit (Lesson 4), plotting.

In [ ]:
NAMES_X = ["Tx", "Mx", "Rx", "mx", "rx"]
NAMES_Y = ["Ty", "My", "Ry", "my", "ry"]


# -----------------------------------------------------------------------------
# 1. Layout
# -----------------------------------------------------------------------------
def make_layout(wafer_r=150.0, edge_excl=3.0, field_w=26.0, field_h=33.0,
                local_targets=((-11, -14), (11, -14), (-11, 14), (11, 14))):
    """Return (centres, data).

    centres : (n_fields, 2) array of field centres (X, Y)
    data    : (n_targets, 7) array with columns
              [field_id, X, Y, x, y, abs_X, abs_Y]
    """
    centres = []
    for i in range(-6, 7):
        for j in range(-5, 6):
            cx, cy = i * field_w, j * field_h
            corners = [(cx + sx * field_w / 2, cy + sy * field_h / 2)
                       for sx in (-1, 1) for sy in (-1, 1)]
            if all(np.hypot(px, py) <= wafer_r - edge_excl for px, py in corners):
                centres.append((cx, cy))
    centres = np.array(centres)

    local = np.array(local_targets, dtype=float)
    rows = [[f, cx, cy, lx, ly, cx + lx, cy + ly]
            for f, (cx, cy) in enumerate(centres)
            for lx, ly in local]
    return centres, np.array(rows)


# -----------------------------------------------------------------------------
# 2. Simulation
# -----------------------------------------------------------------------------
def simulate_overlay(data, true, noise=1.0, n_out=0, out_range=(30, 60), rng=None):
    """Generate measured dx, dy from 'true' parameters + noise + outliers.

    Returns dx, dy and the indices of the damaged (outlier) targets.
    """
    rng = rng if rng is not None else np.random.default_rng()
    X, Y, x, y = data[:, 1], data[:, 2], data[:, 3], data[:, 4]

    dx = true["Tx"] + true["Mx"] * X - true["Rx"] * Y + true["mx"] * x - true["rx"] * y
    dy = true["Ty"] + true["My"] * Y + true["Ry"] * X + true["my"] * y + true["ry"] * x
    dx = dx + rng.normal(0, noise, len(dx))
    dy = dy + rng.normal(0, noise, len(dy))

    bad = rng.choice(len(dx), size=n_out, replace=False) if n_out > 0 else np.array([], int)
    if n_out > 0:
        lo, hi = out_range
        dx[bad] += rng.choice([-1, 1], n_out) * rng.uniform(lo, hi, n_out)
        dy[bad] += rng.choice([-1, 1], n_out) * rng.uniform(lo, hi, n_out)
    return dx, dy, bad


# -----------------------------------------------------------------------------
# 3. Design matrices
# -----------------------------------------------------------------------------
def design_matrices(data):
    """Return (A_x, A_y): one row per target, one column per parameter."""
    X, Y, x, y = data[:, 1], data[:, 2], data[:, 3], data[:, 4]
    ones = np.ones_like(X)
    A_x = np.column_stack([ones, X, -Y, x, -y])   # Tx, Mx, Rx, mx, rx
    A_y = np.column_stack([ones, Y,  X, y,  x])   # Ty, My, Ry, my, ry
    return A_x, A_y


# -----------------------------------------------------------------------------
# 4-5. Fitting
# -----------------------------------------------------------------------------
def fit_plain(A, d):
    """Ordinary least squares using every target."""
    return np.linalg.lstsq(A, d, rcond=None)[0]


def fit_robust(A, d, k=3.0, use_mad=True, max_iter=10):
    """Iterative k-sigma outlier rejection.

    Returns (params, keep) where keep is a boolean mask (True = target used).
    """
    keep = np.ones(len(d), dtype=bool)
    for _ in range(max_iter):
        p = np.linalg.lstsq(A[keep], d[keep], rcond=None)[0]
        res = d - A @ p
        r = res[keep]
        sigma = (1.4826 * np.median(np.abs(r - np.median(r)))) if use_mad else r.std()
        new_keep = np.abs(res) < k * sigma
        if np.array_equal(new_keep, keep):
            break
        keep = new_keep
    return p, keep


# -----------------------------------------------------------------------------
# 6-7. Statistics
# -----------------------------------------------------------------------------
def param_uncertainty(A, d, p):
    """Standard error (1 sigma) of each fitted parameter."""
    res = d - A @ p
    dof = len(d) - A.shape[1]
    sigma = np.sqrt(np.sum(res ** 2) / dof)
    return sigma * np.sqrt(np.diag(np.linalg.inv(A.T @ A)))


def overlay_metric(d):
    """Fab overlay number: |mean| + 3 sigma."""
    return abs(d.mean()) + 3 * d.std()


# -----------------------------------------------------------------------------
# 8. Plotting
# -----------------------------------------------------------------------------
def plot_vector_maps(data, centres, panels, keep=None, wafer_r=150.0,
                     field_w=26.0, field_h=33.0, nm_per_mm=1.5,
                     title="", savepath=None):
    """Draw side-by-side overlay vector maps.

    panels : list of (u, v, subtitle)
    keep   : boolean mask; rejected targets are drawn in red
    """
    keep = np.ones(len(data), bool) if keep is None else keep
    fig, axes = plt.subplots(1, len(panels), figsize=(5.3 * len(panels), 5.5))
    axes = np.atleast_1d(axes)
    for ax, (u, v, sub) in zip(axes, panels):
        ax.add_patch(plt.Circle((0, 0), wafer_r, fill=False))
        for cx, cy in centres:
            ax.add_patch(plt.Rectangle((cx - field_w / 2, cy - field_h / 2),
                                       field_w, field_h, fill=False, lw=0.3, color="0.7"))
        q = ax.quiver(data[keep, 5], data[keep, 6], u[keep], v[keep],
                      angles="xy", scale_units="xy", scale=1 / nm_per_mm, width=0.004)
        if (~keep).any():
            ax.quiver(data[~keep, 5], data[~keep, 6],
                      np.clip(u[~keep], -15, 15), np.clip(v[~keep], -15, 15),
                      color="red", angles="xy", scale_units="xy",
                      scale=1 / nm_per_mm, width=0.005)
            ax.plot(data[~keep, 5], data[~keep, 6], "o", mfc="none", mec="red", ms=9)
        ax.quiverkey(q, 0.78, 1.03, 10, "10 nm", labelpos="E")
        ax.set_aspect("equal")
        ax.set_xlim(-wafer_r - 10, wafer_r + 10)
        ax.set_ylim(-wafer_r - 10, wafer_r + 10)
        ax.set_xlabel("X (mm)")
        ax.set_title(sub)
    axes[0].set_ylabel("Y (mm)")
    if title:
        fig.suptitle(title, y=1.02)
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, dpi=150, bbox_inches="tight")
    return fig

print('Functions loaded.')

## 2. Demo: plain vs robust overlay correction
Change the settings at the top of the cell and re-run it for the exercises.

In [ ]:
# ----------------------------- settings --------------------------------------
SEED, NOISE = 42, 1.0             # random seed, measurement noise (nm, 1 sigma)
N_OUT, OUT_RANGE = 5, (30, 60)    # damaged targets and their size (nm)
K_SIGMA = 3.0                     # outlier rejection limit

TRUE = {"Tx": 3.0, "Ty": -2.0,    # translation (nm)
        "Mx": 0.05, "My": 0.04,   # wafer magnification (ppm)
        "Rx": 0.03, "Ry": 0.02,   # wafer rotation (urad)
        "mx": 0.20, "my": -0.15,  # field magnification (ppm)
        "rx": 0.10, "ry": 0.12}   # field rotation (urad)
# -----------------------------------------------------------------------------

os.makedirs(RESULTS, exist_ok=True)
rng = np.random.default_rng(SEED)

# 1. Layout and simulated measurement
centres, data = make_layout()
dx, dy, bad = simulate_overlay(data, TRUE, NOISE, N_OUT, OUT_RANGE, rng)
A_x, A_y = design_matrices(data)
print(f"Fields: {len(centres)}  Targets: {len(data)}  Damaged: {sorted(bad.tolist())}")

# 2. Plain and robust fits
px_plain, py_plain = fit_plain(A_x, dx), fit_plain(A_y, dy)
px_rob, keep_x = fit_robust(A_x, dx, K_SIGMA)
py_rob, keep_y = fit_robust(A_y, dy, K_SIGMA)
keep = keep_x & keep_y

se_x = param_uncertainty(A_x[keep_x], dx[keep_x], px_rob)
se_y = param_uncertainty(A_y[keep_y], dy[keep_y], py_rob)

# 3. Outlier scorecard
rejected = set(np.where(~keep)[0].tolist())
caught = rejected & set(bad.tolist())
print(f"Outliers caught: {len(caught)}/{N_OUT}   "
      f"false alarms: {sorted(rejected - set(bad.tolist()))}")

# 4. Parameter table
plain = dict(zip(NAMES_X, px_plain)) | dict(zip(NAMES_Y, py_plain))
robust = dict(zip(NAMES_X, px_rob)) | dict(zip(NAMES_Y, py_rob))
se = dict(zip(NAMES_X, se_x)) | dict(zip(NAMES_Y, se_y))

lines = [f"{'term':<5}{'true':>9}{'plain':>10}{'robust':>10}{'±1σ':>9}"]
for k in TRUE:
    lines.append(f"{k:<5}{TRUE[k]:>9.4f}{plain[k]:>10.4f}{robust[k]:>10.4f}{se[k]:>9.4f}")

# 5. Overlay before / after correction
res_x, res_y = dx - A_x @ px_rob, dy - A_y @ py_rob
lines += ["",
          f"Raw overlay      |mean|+3σ: x = {overlay_metric(dx[keep]):.2f} nm, "
          f"y = {overlay_metric(dy[keep]):.2f} nm",
          f"Residual overlay |mean|+3σ: x = {overlay_metric(res_x[keep]):.2f} nm, "
          f"y = {overlay_metric(res_y[keep]):.2f} nm"]
report = "\n".join(lines)
print("\n" + report)
with open(os.path.join(RESULTS, "fit_summary.txt"), "w", encoding="utf-8") as f:
    f.write(report + "\n")

# 6. Plots
plot_vector_maps(
    data, centres,
    [(dx, dy, "Raw (measured)"),
     (A_x @ px_rob, A_y @ py_rob, "Model (robust fit)"),
     (res_x, res_y, "Residual after correction")],
    keep=keep,
    title="Overlay vector maps (red = rejected outlier targets)",
    savepath=os.path.join(RESULTS, "vector_maps.png"))

plt.show()


## 3. Sampling study: which 60 points give the best correction?
Simulates 300 wafers per plan (takes a few seconds).

In [ ]:
N_WAFERS = 300
NOISE = 1.0
TRUE = {"Tx": 3.0, "Ty": -2.0, "Mx": 0.05, "My": 0.04, "Rx": 0.03, "Ry": 0.02,
        "mx": 0.20, "my": -0.15, "rx": 0.10, "ry": 0.12}

os.makedirs(RESULTS, exist_ok=True)
rng = np.random.default_rng(7)

centres, full = make_layout()
radius = np.hypot(centres[:, 0], centres[:, 1])
order = np.argsort(radius)                      # field ids, nearest -> farthest

plans = {
    "All fields (228 pts)":       np.arange(len(centres)),
    "Every 4th field (60 pts)":   np.arange(len(centres))[::4],
    "15 outermost (60 pts)":      order[-15:],
    "14 outer + centre (60 pts)": np.r_[order[-14:], order[0]],
    "15 innermost (60 pts)":      order[:15],
}

results = {}
for name, fields in plans.items():
    data = full[np.isin(full[:, 0], fields)]
    A_x, A_y = design_matrices(data)
    fits = []
    for _ in range(N_WAFERS):
        dx, dy, _ = simulate_overlay(data, TRUE, NOISE, rng=rng)
        fits.append(np.r_[fit_plain(A_x, dx), fit_plain(A_y, dy)])
    results[name] = np.array(fits).std(axis=0)   # scatter of each parameter

terms = NAMES_X + NAMES_Y
show = ["Tx", "Mx", "My", "Rx", "Ry", "mx"]
idx = [terms.index(t) for t in show]

print(f"Parameter scatter (1σ) over {N_WAFERS} simulated wafers\n")
print(f"{'plan':<28}" + "".join(f"{t:>9}" for t in show))
for name, s in results.items():
    print(f"{name:<28}" + "".join(f"{s[i]:>9.4f}" for i in idx))

# Bar chart: wafer magnification Mx scatter per plan
fig, ax = plt.subplots(figsize=(8, 4))
names = list(results)
vals = [results[n][terms.index("Mx")] * 1000 for n in names]
ax.barh(names, vals)
ax.invert_yaxis()
ax.set_xlabel("Mx repeatability, 1σ (ppb)")
ax.set_title("Wafer magnification uncertainty by sampling plan")
for i, v in enumerate(vals):
    ax.text(v, i, f" {v:.2f}", va="center")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, "sampling_study.png"), dpi=150)
plt.show()
